# nb_00b — Set / update pipeline config

Run **after `nb_00_bootstrap`** (which creates the `config` Delta table and seeds placeholder
defaults) to point the pipeline at *your* deployed Azure resources.

1. Attach this notebook to the target lakehouse (`aws_connect_lh`).
2. Edit the `CONFIG` dict below with your endpoints (values from `infra/deploy.ps1` output).
3. Run all. It performs an **upsert** (update existing keys, insert new ones) so it is safe to
   re-run and only changes the keys you list here.

Auth is keyless (Entra ID) — no keys are stored; only endpoints + names go in `config`.

In [ ]:
# ============================ EDIT ME ============================
# Only the keys you list here are changed; everything else in `config` is left as-is.
CONFIG = {
    'doc_intelligence_endpoint': 'https://di-awsconn-v4o42m.cognitiveservices.azure.com/',
    'aoai_endpoint':             'https://aoai-awsconn-v4o42m.openai.azure.com/',
    'aoai_embedding_deployment': 'text-embedding-3-large',
    'search_endpoint':           'https://mmz-ai-search.search.windows.net',
    'search_index_name':         'docs-rag',
    'kv_name':                   'kv-awsconn-v4o42m',
    'search_key_secret':         'search-admin-key',
}
# ================================================================

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import Row
from delta.tables import DeltaTable

now = datetime.now(timezone.utc)
rows = [Row(key=k, value=str(v), value_type='string', updated_utc=now) for k, v in CONFIG.items()]
df = spark.createDataFrame(rows)

# Upsert: update keys that exist, insert keys that don't. Other config rows are untouched.
tgt = DeltaTable.forName(spark, 'config')
(tgt.alias('t')
   .merge(df.alias('s'), 't.key = s.key')
   .whenMatchedUpdateAll()
   .whenNotMatchedInsertAll()
   .execute())

print('Applied', len(CONFIG), 'config keys. Current config:')
spark.table('config').orderBy('key').show(50, truncate=False)